In [1]:
#Modules
%config InlineBackend.figure_format='retina'
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
from astropy.io import fits
import pandas as pd
from io import StringIO
from matplotlib.path import Path
import os
from astropy.wcs import WCS
from astropy import units as u
from astropy.coordinates import SkyCoord
from shapely.geometry import Polygon
from astropy.utils.data import get_pkg_data_filename
from astropy.wcs import Wcsprm
from astropy.io import fits
from astropy.wcs import utils
from skimage.feature import blob_dog, blob_log, blob_doh
from mpl_toolkits.mplot3d import Axes3D
from matplotlib import colors
from skimage.color import rgb2gray, rgb2hsv, hsv2rgb
from skimage.io import imread, imshow
from sklearn.cluster import KMeans
from astropy.visualization import make_lupton_rgb
from astropy.visualization import SqrtStretch,LogStretch
from astropy.visualization import ZScaleInterval
from reproject import reproject_interp
from astropy.visualization import make_lupton_rgb
#import sep
from matplotlib.patches import Ellipse
from scipy import stats
from scipy.ndimage import gaussian_filter
import matplotlib.patches as mpatches
import matplotlib.lines as mlines
from scipy.interpolate import griddata

from scipy.stats import kde
from scipy.ndimage import gaussian_filter
from astropy.io import ascii
from astropy.visualization import simple_norm
from photutils.psf import extract_stars


from astropy.table import Table
from astropy.nddata import NDData

from shapely.geometry import Polygon, Point
from photutils.detection import find_peaks

from skimage.measure import EllipseModel

from astropy.stats import SigmaClip

from photutils.background import Background2D, MedianBackground
from scipy import interpolate
ell = EllipseModel()

image_lims=ZScaleInterval()

plt.rcParams['font.size'] = 15
from scipy.interpolate import LinearNDInterpolator
from scipy.interpolate import CloughTocher2DInterpolator

from astropy.modeling.models import Gaussian2D
from astropy.modeling.models import custom_model
from astropy.modeling import models, fitting
from scipy.optimize import curve_fit

from photutils.segmentation import SegmentationImage
from astropy.stats import sigma_clipped_stats, SigmaClip
from photutils.segmentation import detect_threshold, detect_sources

from astropy.convolution import convolve, Gaussian2DKernel, Gaussian1DKernel
from astropy.convolution import interpolate_replace_nans

from photutils.segmentation import deblend_sources, detect_threshold, detect_sources

from photutils.segmentation import SourceFinder
from photutils.segmentation import SourceCatalog

from astropy.modeling.fitting import LevMarLSQFitter
from astropy.stats import gaussian_sigma_to_fwhm
from photutils.background import MADStdBackgroundRMS, MMMBackground
from photutils.detection import IRAFStarFinder
#from photutils.psf import (DAOGroup, IntegratedGaussianPRF,
 #                          IterativelySubtractedPSFPhotometry,BasicPSFPhotometry,prepare_psf_model)

from photutils.datasets import make_noise_image
from photutils.psf import EPSFBuilder
from photutils.utils import circular_footprint
from tempfile import TemporaryFile
from astropy.io import fits as pyfits
import pickle

In [2]:
#Load Contour Vertices
#Conservative and Expansive Dictionaries use IRAC coordinates
#Conservative Higher and Expansive Higher Dictionaries use VMC coordinates

with open(r"C:\Users\jacob\AstronomyResearchStuff\jw_verts3_conservative.pkl", 'rb') as fp1:
    vertices1 = pickle.load(fp1)
    
with open(r"C:\Users\jacob\AstronomyResearchStuff\verts_conservative_higher_6MARCH.pkl", 'rb') as fp25:
    vertices25 = pickle.load(fp25)
    
with open(r"C:\Users\jacob\AstronomyResearchStuff\jw_verts3_expansive.pkl", 'rb') as fp3:
    vertices3 = pickle.load(fp3)
    
with open(r"C:\Users\jacob\AstronomyResearchStuff\verts_expansive_higher_6MARCH.pkl", 'rb') as fp45:
    vertices45 = pickle.load(fp45)

In [5]:
#Load a VMC file to assist with coordinate conversions
hdux = fits.open(r"C:\Users\jacob\AstronomyResearchStuff\e20191014_00130000213_dp_st_tl.fit") 

In [7]:
#Block of Code Containing the Extended Object Corrections outlined in the IRAC handbook
sourceRadii = 0

extendedFactor1 = 0.82 * np.exp(-1*(sourceRadii**0.37)) + 0.91
extendedFactor2 = 1.16 * np.exp(-1*(sourceRadii**0.433)) + 0.94
extendedFactor3 = 1.49 * np.exp(-1*(sourceRadii**0.207)) + 0.66
extendedFactor4 = 1.37 * np.exp(-1*(sourceRadii**0.33)) + 0.74

In [11]:
#Open IRAC Mosaics
hdu1 = fits.open(r"C:\Users\jacob\AstronomyResearchStuff\SAGE_LMC_IRAC3.6_1.2_mosaic.fits")
hdu2 = fits.open(r"C:\Users\jacob\AstronomyResearchStuff\SAGE_LMC_IRAC4.5_1.2_mosaic.fits")
hdu3 = fits.open(r"C:\Users\jacob\AstronomyResearchStuff\SAGE_LMC_IRAC5.8_1.2_mosaic.fits")
hdu4 = fits.open(r"C:\Users\jacob\AstronomyResearchStuff\SAGE_LMC_IRAC8.0_1.2_mosaic.fits")

In [13]:
#Set necessary constants and conversions
zpt1 = 277.5
zpt2 = 179.5
zpt3 = 116.5
zpt4 = 63.13
pix2arcsec = 1.2
arcsec2degree = 1./3600.
sr2squaredegree = 4.*np.pi/41253.
MJy2Jy = 1000000
Jy2Wms = 10**-26

In [15]:
def nonHigherPhotometry(catalog):

    #Computes the 3.6, 4.5, 5.8, and 8.0 micron photometric magnitudes for non-higher sets of vertices

    #Inputs
    #---------

    #catalog: dict
        #dictionary of contour vertices to calculate the photometric magnitudes of

    #Set a variable to cycle through the photometry wavelengths
    ran = [0,1,2,3]
    
    #Create a blank dictionary to store magnitudes
    magnitudes2 = {}
    magnitudes2['key'] = []
    magnitudes2['3.6'] = []
    magnitudes2['4.5'] = []
    magnitudes2['5.8'] = []
    magnitudes2['8.0'] = []

    #Iterates over the wavelengths to compute photometries
    for ii in ran:
        if ii == 0:
            
            #3.6 Micron and Keys
            #Load 3.6 micron image
            backsub1 = np.load(r"C:\Users\jacob\OneDrive\Desktop\Cluster_Project\36_background_subtracted.npy")

            #Iterate over all keys in the vertices catalog
            for key in catalog:

                array = catalog.get(key)   

                ra = array[0]
                dec = array[1]

                magnitudes2['key'].append(key)


                print('These are the 3.6 magnitudes of Contour ' + str(key))

                #Set the boundaries around the isophote
                xpixels, ypixels = WCS(hdu1[0].header).all_world2pix(ra,dec, 1)
                xmin = np.min(xpixels)-1
                xmax = np.max(xpixels)+1
                ymin = np.min(ypixels)-1
                ymax = np.max(ypixels)+1


                #3.6 Micron Magnitude
                #Sum the light flux values of the pixels within the isophote
                cutout = backsub1[int(np.round(ymin)-1):int(np.round(ymax)-1), int(np.round(xmin)-1):int(np.round(xmax)-1)]
                flat = cutout.flatten()
                xv,yv = np.indices(cutout.shape)
                flaty = xv.flatten()
                flatx = yv.flatten()
                points = np.vstack((flatx,flaty)).T
                polygon = Path(np.stack((xpixels-xmin, ypixels-ymin), axis = -1 ))
                grid = polygon.contains_points(points)
                index_true, = np.where(grid == True)
                index2 = np.reshape(grid, cutout.shape)
                copy = cutout.copy()
                copy[index2] = -9000
                sumAll = sum(cutout.flatten())

                sumInside = sum(cutout[index2])

                #This section finds the object radius and then determines if the object correction is needed
                vertices = np.stack((xpixels, ypixels),axis = 1)
                polygon = Polygon(vertices)
                area = polygon.area
                radius = np.sqrt(area/np.pi)
                radius = radius*1.2


                if radius >= 7:
                    extendedFactor1 = 0.82 * np.exp(-1*(radius**0.37)) + 0.91
                else:
                    extendedFactor1 = 1

                #Perform the unit conversions on the magnitude and apply the extended object correction factor
                initialFluxDensity1 = sumInside
                initialFluxDensity1 = initialFluxDensity1*extendedFactor1
                convertedFluxDensity1 = initialFluxDensity1*pix2arcsec**2*arcsec2degree**2*sr2squaredegree*MJy2Jy   
                convertedFluxDensity1 = float(convertedFluxDensity1)
                mag1 = (2.5*np.log10(zpt1/convertedFluxDensity1))  

                
                print(mag1)

                magnitudes2['3.6'].append(mag1)

                


        elif ii == 1:

        #4.5 Micron

           
            #Load 4.5 micron image
            del backsub1
            backsub2 = np.load(r"C:\Users\jacob\OneDrive\Desktop\Cluster_Project\45_background_subtracted.npy")

            for key in catalog:

                array = catalog.get(key)   

                ra = array[0]
                dec = array[1]

                print('These are the 4.5 magnitudes of Contour ' + str(key))

                #Set the boundaries around the isophote
                xpixels, ypixels = WCS(hdu2[0].header).all_world2pix(ra,dec, 1)
                xmin = np.min(xpixels)-1
                xmax = np.max(xpixels)+1
                ymin = np.min(ypixels)-1
                ymax = np.max(ypixels)+1

                #4.5 Micron Magnitude
                #Sum the light flux values of the pixels within the isophote
                cutout = backsub2[int(np.round(ymin)-1):int(np.round(ymax)-1), int(np.round(xmin)-1):int(np.round(xmax)-1)]
                flat = cutout.flatten()
                xv,yv = np.indices(cutout.shape)
                flaty = xv.flatten()
                flatx = yv.flatten()
                points = np.vstack((flatx,flaty)).T
                polygon = Path(np.stack((xpixels-xmin, ypixels-ymin), axis = -1 ))
                grid = polygon.contains_points(points)
                index_true, = np.where(grid == True)
                index2 = np.reshape(grid, cutout.shape)
                copy = cutout.copy()
                copy[index2] = -9000
                sumAll = sum(cutout.flatten())

                sumInside = sum(cutout[index2])

                #This section finds the object radius and then determines if the object correction is needed
                vertices = np.stack((xpixels, ypixels),axis = 1)
                polygon = Polygon(vertices)
                area = polygon.area
                radius = np.sqrt(area/np.pi)
                radius = radius*1.2

                if radius >= 7:
                    extendedFactor2 = 1.16 * np.exp(-1*(radius**0.433)) + 0.94
                else:
                    extendedFactor2 = 1

                #Perform the unit conversions on the magnitude and apply the extended object correction factor
                initialFluxDensity2 = sumInside
                initialFluxDensity2 = initialFluxDensity2*extendedFactor2
                convertedFluxDensity2 = initialFluxDensity2*pix2arcsec**2*arcsec2degree**2*sr2squaredegree*MJy2Jy   
                convertedFluxDensity2 = float(convertedFluxDensity2)
                mag2 = (2.5*np.log10(zpt2/convertedFluxDensity2))  
                print(mag2)

                magnitudes2['4.5'].append(mag2)

                

        elif ii == 2:

            #5.8 Micron
            #Load 5.8 micron image
            del backsub2 
            backsub3 = np.load(r"C:\Users\jacob\OneDrive\Desktop\Cluster_Project\58_background_subtracted.npy")



            for key in catalog:


                array = catalog.get(key)   
                
                ra = array[0]
                dec = array[1]
                print('These are the 5.8 magnitudes of Contour ' + str(key))

                #Set the boundaries around the isophote
                xpixels, ypixels = WCS(hdu3[0].header).all_world2pix(ra,dec, 1)
                xmin = np.min(xpixels)-1
                xmax = np.max(xpixels)+1
                ymin = np.min(ypixels)-1
                ymax = np.max(ypixels)+1



                #5.8 Micron Magnitude
                #Sum the light flux values of the pixels within the isophote
                cutout = backsub3[int(np.round(ymin)-1):int(np.round(ymax)-1), int(np.round(xmin)-1):int(np.round(xmax)-1)]
                flat = cutout.flatten()
                xv,yv = np.indices(cutout.shape)
                flaty = xv.flatten()
                flatx = yv.flatten()
                points = np.vstack((flatx,flaty)).T
                polygon = Path(np.stack((xpixels-xmin, ypixels-ymin), axis = -1 ))
                grid = polygon.contains_points(points)
                index_true, = np.where(grid == True)
                index2 = np.reshape(grid, cutout.shape)
                copy = cutout.copy()
                copy[index2] = -9000
                sumAll = sum(cutout.flatten())

                sumInside = sum(cutout[index2])

                #This section finds the object radius and then determines if the object correction is needed
                vertices = np.stack((xpixels, ypixels),axis = 1)
                polygon = Polygon(vertices)
                area = polygon.area
                radius = np.sqrt(area/np.pi)
                radius = radius*1.2


                if radius >= 7:
                    extendedFactor3 = 1.49 * np.exp(-1*(radius**0.207)) + 0.66
                else:
                    extendedFactor3 = 1

                #Perform the unit conversions on the magnitude and apply the extended object correction factor
                initialFluxDensity3 = sumInside
                initialFluxDensity3 = initialFluxDensity3*extendedFactor3
                convertedFluxDensity3 = initialFluxDensity3*pix2arcsec**2*arcsec2degree**2*sr2squaredegree*MJy2Jy   
                convertedFluxDensity3 = float(convertedFluxDensity3)
                mag3 = (2.5*np.log10(zpt3/convertedFluxDensity3))  
                print(mag3)

                magnitudes2['5.8'].append(mag3)

             

        elif ii == 3:

            #8.0 Micron
            #Load 8.0 micron image
            del backsub3 
            backsub4 = np.load(r"C:\Users\jacob\OneDrive\Desktop\Cluster_Project\80_background_subtracted.npy")


            

            for key in catalog:


                array = catalog.get(key) 

                ra = array[0]
                dec = array[1]

             

                print('These are the 8.0 magnitudes of Contour ' + str(key))

                #Set the boundaries around the isophote
                xpixels, ypixels = WCS(hdu4[0].header).all_world2pix(ra,dec, 1)
                xmin = np.min(xpixels)-1
                xmax = np.max(xpixels)+1
                ymin = np.min(ypixels)-1
                ymax = np.max(ypixels)+1



                #8.0 Micron Magnitude
                #Sum the light flux values of the pixels within the isophote
                cutout = backsub4[int(np.round(ymin)-1):int(np.round(ymax)-1), int(np.round(xmin)-1):int(np.round(xmax)-1)]
                flat = cutout.flatten()
                xv,yv = np.indices(cutout.shape)
                flaty = xv.flatten()
                flatx = yv.flatten()
                points = np.vstack((flatx,flaty)).T
                polygon = Path(np.stack((xpixels-xmin, ypixels-ymin), axis = -1 ))
                grid = polygon.contains_points(points)
                index_true, = np.where(grid == True)
                index2 = np.reshape(grid, cutout.shape)
                copy = cutout.copy()
                copy[index2] = -9000
                sumAll = sum(cutout.flatten())

                sumInside = sum(cutout[index2])

                #This section finds the object radius and then determines if the object correction is needed
                vertices = np.stack((xpixels, ypixels),axis = 1)
                polygon = Polygon(vertices)
                area = polygon.area
                radius = np.sqrt(area/np.pi)
                radius = radius*1.2


                if radius >= 7:
                    extendedFactor4 = 1.37 * np.exp(-1*(radius**0.33)) + 0.74
                else:
                    extendedFactor4 = 1

                #Perform the unit conversions on the magnitude and apply the extended object correction factor
                initialFluxDensity4 = sumInside
                initialFluxDensity4 = initialFluxDensity4*extendedFactor4
                convertedFluxDensity4 = initialFluxDensity4*pix2arcsec**2*arcsec2degree**2*sr2squaredegree*MJy2Jy   
                convertedFluxDensity4 = float(convertedFluxDensity4)
                mag4 = (2.5*np.log10(zpt4/convertedFluxDensity4))  
                print(mag4)

                magnitudes2['8.0'].append(mag4)

            del backsub4

            #Create a table using the magnitude dictionary and save as a FITS file

            magnitudes2_table = Table(magnitudes2)
            print(magnitudes2_table)
            magnitudes2_table.write('Spitzer_magnitudes2__ECF.fits')



In [ ]:
#Compute the conservative photometry
catalog = vertices1
nonHigherPhotometry(catalog)

In [ ]:
#Compute the expansive photometry
catalog = vertices3
nonHigherPhotometry(catalog)

In [23]:
def higherPhotometry(catalog):

    #Computes the 3.6, 4.5, 5.8, and 8.0 micron photometric magnitudes for higher sets of vertices

    #Inputs
    #---------

    #catalog: dict
        #dictionary of contour vertices to calculate the photometric magnitudes of


    #Set a variable to iterate through the photometric wavelengths
    ran = [0,1,2,3]
    
    #Create a blank dictionary to store magnitudes
    magnitudes2 = {}
    magnitudes2['key'] = []
    magnitudes2['3.6'] = []
    magnitudes2['4.5'] = []
    magnitudes2['5.8'] = []
    magnitudes2['8.0'] = []


    #Iterates over wavelengths to compute photometric magnitudes
    for ii in ran:
        if ii == 0:
            

            #3.6 Micron and Keys
            #Load 3.6 micron image
            backsub1 = np.load(r"C:\Users\jacob\OneDrive\Desktop\Cluster_Project\36_background_subtracted.npy")

            for key in catalog:

                array = catalog.get(key)

                pixx1 = array[:,0]
                pixy1 = array[:,1]

                ra, dec = WCS(hdux[1].header).all_pix2world(pixx1,pixy1, 1)


                magnitudes2['key'].append(key)


                print('These are the 3.6 magnitudes of Contour ' + str(key))

                #Set the boundaries around the isophote
                xpixels, ypixels = WCS(hdu1[0].header).all_world2pix(ra,dec, 1)
                xmin = np.min(xpixels)-1
                xmax = np.max(xpixels)+1
                ymin = np.min(ypixels)-1
                ymax = np.max(ypixels)+1


                #3.6 Micron Magnitude
                #Sum the light flux values of the pixels within the isophote
                cutout = backsub1[int(np.round(ymin)-1):int(np.round(ymax)-1), int(np.round(xmin)-1):int(np.round(xmax)-1)]
                flat = cutout.flatten()
                xv,yv = np.indices(cutout.shape)
                flaty = xv.flatten()
                flatx = yv.flatten()
                points = np.vstack((flatx,flaty)).T
                polygon = Path(np.stack((xpixels-xmin, ypixels-ymin), axis = -1 ))
                grid = polygon.contains_points(points)
                index_true, = np.where(grid == True)
                index2 = np.reshape(grid, cutout.shape)
                copy = cutout.copy()
                copy[index2] = -9000
                sumAll = sum(cutout.flatten())

                sumInside = sum(cutout[index2])

                #This section finds the object radius and then determines if the object correction is needed
                vertices = np.stack((xpixels, ypixels),axis = 1)
                polygon = Polygon(vertices)
                area = polygon.area
                radius = np.sqrt(area/np.pi)
                radius = radius*1.2


                if radius >= 7:
                    extendedFactor1 = 0.82 * np.exp(-1*(radius**0.37)) + 0.91
                else:
                    extendedFactor1 = 1

                #Perform the unit conversions on the magnitude and apply the extended object correction factor
                initialFluxDensity1 = sumInside
                initialFluxDensity1 = initialFluxDensity1*extendedFactor1
                convertedFluxDensity1 = initialFluxDensity1*pix2arcsec**2*arcsec2degree**2*sr2squaredegree*MJy2Jy   
                convertedFluxDensity1 = float(convertedFluxDensity1)
                mag1 = (2.5*np.log10(zpt1/convertedFluxDensity1))  
                print(mag1)

                magnitudes2['3.6'].append(mag1)

                

        elif ii == 1:

        #4.5 Micron

           
            #Load 4.5 micron image
            del backsub1
            backsub2 = np.load(r"C:\Users\jacob\OneDrive\Desktop\Cluster_Project\45_background_subtracted.npy")

            for key in catalog:

                array = catalog.get(key)

                pixx1 = array[:,0]
                pixy1 = array[:,1]

                ra, dec = WCS(hdux[1].header).all_pix2world(pixx1,pixy1, 1)

                print('These are the 4.5 magnitudes of Contour ' + str(key))

                #Set the boundaries around the isophote
                xpixels, ypixels = WCS(hdu2[0].header).all_world2pix(ra,dec, 1)
                xmin = np.min(xpixels)-1
                xmax = np.max(xpixels)+1
                ymin = np.min(ypixels)-1
                ymax = np.max(ypixels)+1

                #4.5 Micron Magnitude
                #Sum the light flux values of the pixels within the isophote
                cutout = backsub2[int(np.round(ymin)-1):int(np.round(ymax)-1), int(np.round(xmin)-1):int(np.round(xmax)-1)]
                flat = cutout.flatten()
                xv,yv = np.indices(cutout.shape)
                flaty = xv.flatten()
                flatx = yv.flatten()
                points = np.vstack((flatx,flaty)).T
                polygon = Path(np.stack((xpixels-xmin, ypixels-ymin), axis = -1 ))
                grid = polygon.contains_points(points)
                index_true, = np.where(grid == True)
                index2 = np.reshape(grid, cutout.shape)
                copy = cutout.copy()
                copy[index2] = -9000
                sumAll = sum(cutout.flatten())

                sumInside = sum(cutout[index2])

                #This section finds the object radius and then determines if the object correction is needed
                vertices = np.stack((xpixels, ypixels),axis = 1)
                polygon = Polygon(vertices)
                area = polygon.area
                radius = np.sqrt(area/np.pi)
                radius = radius*1.2

                if radius >= 7:
                    extendedFactor2 = 1.16 * np.exp(-1*(radius**0.433)) + 0.94
                else:
                    extendedFactor2 = 1

                
                #Perform the unit conversions on the magnitude and apply the extended object correction factor
                initialFluxDensity2 = sumInside
                initialFluxDensity2 = initialFluxDensity2*extendedFactor2
                convertedFluxDensity2 = initialFluxDensity2*pix2arcsec**2*arcsec2degree**2*sr2squaredegree*MJy2Jy   
                convertedFluxDensity2 = float(convertedFluxDensity2)
                mag2 = (2.5*np.log10(zpt2/convertedFluxDensity2))  
                print(mag2)

                
                magnitudes2['4.5'].append(mag2)

               
        elif ii == 2:

            #5.8 Micron
            #Load 5.8 micron image
            del backsub2 
            backsub3 = np.load(r"C:\Users\jacob\OneDrive\Desktop\Cluster_Project\58_background_subtracted.npy")


           

            for key in catalog:


                array = catalog.get(key)

                pixx1 = array[:,0]
                pixy1 = array[:,1]

                ra, dec = WCS(hdux[1].header).all_pix2world(pixx1,pixy1, 1)

                print('These are the 5.8 magnitudes of Contour ' + str(key))

                #Set the boundaries around the isophote
                xpixels, ypixels = WCS(hdu3[0].header).all_world2pix(ra,dec, 1)
                xmin = np.min(xpixels)-1
                xmax = np.max(xpixels)+1
                ymin = np.min(ypixels)-1
                ymax = np.max(ypixels)+1



                #5.8 Micron Magnitude
                #Sum the light flux values of the pixels within the isophote
                cutout = backsub3[int(np.round(ymin)-1):int(np.round(ymax)-1), int(np.round(xmin)-1):int(np.round(xmax)-1)]
                flat = cutout.flatten()
                xv,yv = np.indices(cutout.shape)
                flaty = xv.flatten()
                flatx = yv.flatten()
                points = np.vstack((flatx,flaty)).T
                polygon = Path(np.stack((xpixels-xmin, ypixels-ymin), axis = -1 ))
                grid = polygon.contains_points(points)
                index_true, = np.where(grid == True)
                index2 = np.reshape(grid, cutout.shape)
                copy = cutout.copy()
                copy[index2] = -9000
                sumAll = sum(cutout.flatten())

                sumInside = sum(cutout[index2])

                #This section finds the object radius and then determines if the object correction is needed
                vertices = np.stack((xpixels, ypixels),axis = 1)
                polygon = Polygon(vertices)
                area = polygon.area
                radius = np.sqrt(area/np.pi)
                radius = radius*1.2


                if radius >= 7:
                    extendedFactor3 = 1.49 * np.exp(-1*(radius**0.207)) + 0.66
                else:
                    extendedFactor3 = 1


                #Perform the unit conversions on the magnitude and apply the extended object correction factor
                initialFluxDensity3 = sumInside
                initialFluxDensity3 = initialFluxDensity3*extendedFactor3
                convertedFluxDensity3 = initialFluxDensity3*pix2arcsec**2*arcsec2degree**2*sr2squaredegree*MJy2Jy   
                convertedFluxDensity3 = float(convertedFluxDensity3)
                mag3 = (2.5*np.log10(zpt3/convertedFluxDensity3))  
                print(mag3)

                magnitudes2['5.8'].append(mag3)

               

        elif ii == 3:

            #8.0 Micron
            #Load 8.0 micron image
            del backsub3 
            backsub4 = np.load(r"C:\Users\jacob\OneDrive\Desktop\Cluster_Project\80_background_subtracted.npy")


          

            for key in catalog:


                array = catalog.get(key)

                pixx1 = array[:,0]
                pixy1 = array[:,1]

                ra, dec = WCS(hdux[1].header).all_pix2world(pixx1,pixy1, 1)

                print('These are the 8.0 magnitudes of Contour ' + str(key))

                #Set the boundaries around the isophote
                xpixels, ypixels = WCS(hdu4[0].header).all_world2pix(ra,dec, 1)
                xmin = np.min(xpixels)-1
                xmax = np.max(xpixels)+1
                ymin = np.min(ypixels)-1
                ymax = np.max(ypixels)+1



                #8.0 Micron Magnitude
                #Sum the light flux values of the pixels within the isophote
                cutout = backsub4[int(np.round(ymin)-1):int(np.round(ymax)-1), int(np.round(xmin)-1):int(np.round(xmax)-1)]
                flat = cutout.flatten()
                xv,yv = np.indices(cutout.shape)
                flaty = xv.flatten()
                flatx = yv.flatten()
                points = np.vstack((flatx,flaty)).T
                polygon = Path(np.stack((xpixels-xmin, ypixels-ymin), axis = -1 ))
                grid = polygon.contains_points(points)
                index_true, = np.where(grid == True)
                index2 = np.reshape(grid, cutout.shape)
                copy = cutout.copy()
                copy[index2] = -9000
                sumAll = sum(cutout.flatten())

                sumInside = sum(cutout[index2])

                #This section finds the object radius and then determines if the object correction is needed
                vertices = np.stack((xpixels, ypixels),axis = 1)
                polygon = Polygon(vertices)
                area = polygon.area
                radius = np.sqrt(area/np.pi)
                radius = radius*1.2


                if radius >= 7:
                    extendedFactor4 = 1.37 * np.exp(-1*(radius**0.33)) + 0.74
                else:
                    extendedFactor4 = 1


                #Perform the unit conversions on the magnitude and apply the extended object correction factor
                initialFluxDensity4 = sumInside
                initialFluxDensity4 = initialFluxDensity4*extendedFactor4
                convertedFluxDensity4 = initialFluxDensity4*pix2arcsec**2*arcsec2degree**2*sr2squaredegree*MJy2Jy   
                convertedFluxDensity4 = float(convertedFluxDensity4)
                mag4 = (2.5*np.log10(zpt4/convertedFluxDensity4))  
                print(mag4)

                magnitudes2['8.0'].append(mag4)

                

            del backsub4

            #Create a table using the magnitude dictionary and save as a FITS file

            magnitudes2_table = Table(magnitudes2)
            print(magnitudes2_table)
            magnitudes2_table.write('Spitzer_magnitudes2_new_higher_ECF.fits')



In [ ]:
#Compute the conservative higher photometry
catalog = vertices25
higherPhotometry(catalog)

In [ ]:
#Compute the expansive higher photometry
catalog = vertices45
higherPhotometry(catalog)